In [7]:
# Import library require for process
import pandas as pd
from sklearn.model_selection import train_test_split 
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
import warnings as ws
ws.filterwarnings('ignore')



def feature_importance(indep_X, dep_Y, n=5):
    models = [
        ("LinearRegression", LinearRegression()),
        ("Ridge", Ridge()),
        ("Lasso", Lasso(alpha=0.1)),
        ("DecisionTreeRegressor", DecisionTreeRegressor(random_state=0)),
        ("SVR", SVR(kernel="linear")),
        ("RandomForestRegressor", RandomForestRegressor(n_estimators=100, random_state=42)),
    ]

    rfelist = []

    for name, model in models:
        #trains the model using your features indep_X and target dep_Y.
        model.fit(indep_X, dep_Y)

        #Checks whether the model provides built-in feature importance scores (used by tree-based models like Decision Trees and Random Forests).
        
        if hasattr(model, "feature_importances_"):
            importances = pd.Series(model.feature_importances_, index=indep_X.columns)
            
        #Checks whether the model provides coefficients, which are used by linear models such as Linear Regression, Ridge, and Lasso.
        elif hasattr(model, "coef_"):
            
            #gets the learned coefficients from the model.
            coef = model.coef_

            #Converts a 2D coefficient array into a 1D array to match the feature list.
            if coef.ndim > 1:
                coef = coef.ravel()
                #Converts the absolute coefficients into a labeled Series with an importance score for each feature.
                importances = pd.Series(abs(coef), index=indep_X.columns)
        # Handles models without built-in feature importance or coefficients (e.g., SVR).
        else:
            #Computes feature importance by measuring the performance drop after shuffling each feature.
            perm = permutation_importance(model, indep_X, dep_Y, n_repeats=10, random_state=42)
            importances = pd.Series(perm.importances_mean, index=indep_X.columns)

        # #Sorts features by importance and selects the top `n` feature names.
        selected_cols = importances.sort_values(ascending=False).head(n).index.tolist()
        #Creates a new DataFrame with only the top selected features.
        selected_features = indep_X[selected_cols]
        rfelist.append((name, selected_features))

    return rfelist

  
#split_scalar - Split the input, output train and test set. then changes the input to scalar value    
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test

# r2_prediction - used for regression method, model prediction evaluate method
def r2_prediction(regressor,X_test,y_test):
     y_pred = regressor.predict(X_test)
     from sklearn.metrics import r2_score
     r2=r2_score(y_test,y_pred)
     return r2
    
# Linear method is used for Linear regression model creation and r2 prediction
def fit_linear(X_train,y_train,X_test):       
        regressor = LinearRegression()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2   

# Ridge method is used for svm model creation and r2 prediction
def fit_ridge(X_train,y_train,X_test):                
        regressor = Ridge()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
    
# Lasso method is used for svm_NL model creation and r2 prediction   
def fit_lasso(X_train,y_train,X_test):                
        regressor = Lasso(alpha=0.1)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     
# Decision method is used for Decision tree model creation and r2 prediction   
def fit_decision(X_train,y_train,X_test):
        regressor = DecisionTreeRegressor(random_state=0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     
# SVR method is used for random forest model creation and r2 prediction  
def fit_svr(X_train,y_train,X_test):       
        regressor = SVR(kernel='linear')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2 
# random method is used for random forest model creation and r2 prediction  
def fit_random(X_train,y_train,X_test):       
        regressor = RandomForestRegressor(n_estimators=100, random_state=42)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2 
    
# RFE_regression method is used for create dataset with columns name as 'Linear','SVMl','SVMnl','Decision','Random' and index ChiSquare
# and fill the each columns values 
def Feature_regression(acclin,accsRid,accLa,accdes,accrSvR,accrf): 
    
    dataframe=pd.DataFrame(index=['Linear', 'Ridge', 'Lasso', 'Decision', 'SVR', 'Random'],columns=['Linear','Ridge','Lasso','Decision', 'SVR', 'Random'
                                                                                     ])

    for number,idex in enumerate(dataframe.index):
        
        dataframe['Linear'][idex]=acclin[number]       
        dataframe['Ridge'][idex]=accsRid[number]
        dataframe['Lasso'][idex]=accLa[number]
        dataframe['Decision'][idex]=accdes[number]
        dataframe['SVR'][idex]=accrSvR[number]
        dataframe['Random'][idex]=accrf[number]
    return dataframe
    

In [14]:
# Read data from file and datatset should without index
dataset=pd.read_csv("prep.csv",index_col=None)
df2=dataset
# Preprocessed by one hot encoding
df2 = pd.get_dummies(df2, drop_first=True)
# assign the input only
indep_X=df2.drop('classification_yes', axis=1)
# assign output only
dep_Y=df2['classification_yes']

# choose the feature selection here using n feature 
FIMList=feature_importance(indep_X,dep_Y,10)      


In [15]:
# Create 5 empty list for each algorithm and split the input and output
# Evalute each algorithmwise r2 score and send RFE_regression funtion
# finally the evalution data represent by table view.
acclin=[]
accsRid=[]
accLa=[]
accdes=[]
accrf=[]
accrSvR=[]
accrf =[]
index_labels = []


for name, features_df in FIMList:   
    print(name, features_df.shape)
    index_labels.append(name) 
    
    X_train, X_test, y_train, y_test=split_scalar(features_df,dep_Y)  
    r2_lin=fit_linear(X_train,y_train,X_test)
    acclin.append(r2_lin)
    
    r2_ri=fit_ridge(X_train,y_train,X_test)    
    accsRid.append(r2_ri)
    
    r2_la=fit_lasso(X_train,y_train,X_test)
    accLa.append(r2_la)
    
    r2_d=fit_decision(X_train,y_train,X_test)
    accdes.append(r2_d)
    
    r2_sr=fit_svr(X_train,y_train,X_test)
    accrSvR.append(r2_sr)

    r2_r=fit_random(X_train,y_train,X_test)
    accrf.append(r2_r)
    
    
result=Feature_regression(acclin,accsRid,accLa,accdes,accrSvR,accrf)

LinearRegression (399, 10)
Ridge (399, 10)
Lasso (399, 10)
DecisionTreeRegressor (399, 10)
SVR (399, 10)
RandomForestRegressor (399, 10)


In [16]:
result
# 10

,Linear,Ridge,Lasso,Decision,SVR,Random
Linear,0.714887,0.714957,0.599769,0.968654,0.690464,0.96383
Ridge,0.714887,0.714957,0.599769,0.968654,0.690464,0.96383
Lasso,0.60303,0.603111,0.538828,0.826389,0.556655,0.84138
Decision,0.696777,0.697212,0.581409,0.782986,0.653055,0.910326
SVR,0.662764,0.66281,0.594996,0.923777,0.643035,0.942185
Random,0.713361,0.713391,0.581403,0.826389,0.680718,0.915686
